# TP2 — Nettoyer des données pas très propres

**Cas d'usage 01 — Certification IA : Résiliation client SaaS (churn)**

| | |
|---|---|
| **Objectif** | Rendre les données réellement exploitables : doublons, dates multi-formats, nombres stockés en texte, catégorielles mal normalisées, valeurs manquantes. |
| **Livrable** | Un jeu de données propre, documenté, chargé dans une **table PostgreSQL** dédiée (`churn_saas_db`). |
| **Enjeu** | Les dates sont écrites de 3 façons différentes, les nombres ont des virgules à la place des points, certaines catégories sont corrompues par un problème d'encodage — comment tout uniformiser sans perdre ni fausser l'information ? |
| **Compétences** | C3 (préparer les données) |

> Suite de `TP1.ipynb` (cadrage). Décision actée : PostgreSQL est introduit à partir de
> ce TP (base séparée `churn_saas_db`, indépendante de `indusense_db` utilisée par le
> projet `ML/`) — voir `progression_pedagogique.md`.

## §0 — Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 40)

df_raw = pd.read_csv("churn_saas_complet.csv", dtype=str)
df_catalogue = pd.read_csv("catalogue_plans.csv")

print(f"Chargé : {df_raw.shape[0]:,} lignes x {df_raw.shape[1]} colonnes (tout en texte, pour un nettoyage contrôlé)")


Chargé : 5,035 lignes x 29 colonnes (tout en texte, pour un nettoyage contrôlé)


## §1 — Détection et suppression des doublons [C3]

In [2]:
n_dup_full = df_raw.duplicated().sum()
n_dup_id   = df_raw.duplicated(subset=["client_id"]).sum()
print(f"Doublons stricts (toutes colonnes identiques) : {n_dup_full}")
print(f"Doublons sur client_id                        : {n_dup_id}")

df = df_raw.drop_duplicates().reset_index(drop=True)
print(f"\nAvant : {len(df_raw):,} lignes  ->  Après : {len(df):,} lignes  (-{len(df_raw)-len(df)})")


Doublons stricts (toutes colonnes identiques) : 35
Doublons sur client_id                        : 35

Avant : 5,035 lignes  ->  Après : 5,000 lignes  (-35)


Les doublons détectés sont des **doublons stricts** (lignes entièrement identiques,
`client_id` compris) — suppression directe, sans arbitrage nécessaire sur laquelle
conserver.

## §2 — Dates multi-formats [C3]

Trois formats coexistent dans `date_souscription` : ISO (`2024-03-06`), jour/mois/année
(`04/03/2024`) et jour + mois abrégé anglais + année (`16 Apr 2023`). Un simple
`pd.to_datetime` sans format explicite devine mal ce mélange — on tente une liste de
formats connus, dans l'ordre, plutôt que de laisser l'inférence automatique se tromper
silencieusement.

In [3]:
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d %b %Y"]

def parse_date_multi(series, formats):
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    still_missing = series.notna() & result.isna()
    for fmt in formats:
        parsed = pd.to_datetime(series.where(still_missing), format=fmt, errors="coerce")
        result = result.fillna(parsed)
        still_missing = series.notna() & result.isna()
    return result

df["date_souscription"] = parse_date_multi(df["date_souscription"], DATE_FORMATS)

n_non_parsees = df["date_souscription"].isna().sum()
print(f"Dates non parsées après les 3 formats connus : {n_non_parsees} / {len(df)}")
print(df["date_souscription"].sample(8, random_state=1).sort_values())


Dates non parsées après les 3 formats connus : 0 / 5000
4767   2023-07-13
2701   2024-03-02
3499   2024-05-01
1179   2024-08-07
3814   2024-08-29
2735   2024-10-24
2764   2024-11-02
3922   2025-01-06
Name: date_souscription, dtype: datetime64[ns]


## §3 — Nombres stockés en texte [C3]

Quatre colonnes numériques mélangent virgule décimale (`"83,3"`) et point (`"80.0"`),
avec des suffixes hétérogènes : `%` sur `taux_adoption_pct`, `€` sur
`revenu_mensuel_recurrent_eur`, `h` sur `delai_reponse_support_h`. Un premier passage
qui ne retirait que `%` a laissé passer `€`/`h` sans erreur apparente — `pd.to_numeric`
échouait silencieusement sur ces valeurs et les convertissait en `NaN`, gonflant le taux
de valeurs manquantes sans qu'aucune exception ne le signale. Plutôt que d'énumérer
chaque symbole vu jusqu'ici (et risquer d'en manquer un autre), on **extrait le token
numérique en tête de chaîne** par une expression régulière — robuste à tout suffixe non
anticipé.

In [4]:
NUMERIC_TEXT_COLS = [
    "taux_adoption_pct", "revenu_mensuel_recurrent_eur",
    "heures_usage_30j", "delai_reponse_support_h",
]

def clean_numeric_text(series):
    # virgule décimale -> point, puis on extrait le token numérique en tête de
    # chaîne : couvre à la fois les suffixes connus ("%", "€", "h") et tout
    # symbole non anticipé, sans avoir à les énumérer un par un.
    cleaned = series.str.strip().str.replace(",", ".", regex=False)
    numeric_token = cleaned.str.extract(r"(-?\d+\.?\d*)")[0]
    return pd.to_numeric(numeric_token, errors="coerce")

for col in NUMERIC_TEXT_COLS:
    df[col] = clean_numeric_text(df[col])

# Colonnes déjà numériques "propres" (entiers stockés en texte, sans virgule ni %)
CLEAN_INT_COLS = [
    "anciennete_mois", "sieges_souscrits", "utilisateurs_actifs", "connexions_30j",
    "fonctionnalites_total", "fonctionnalites_utilisees", "nb_integrations",
    "derniere_connexion_jours", "tickets_support_90j", "csat",
    "retards_paiement_12m", "sante_compte_fin_periode", "valeur_vie_client_eur", "churn",
]
for col in CLEAN_INT_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[NUMERIC_TEXT_COLS + ["churn"]].dtypes)
print()
print(df[NUMERIC_TEXT_COLS].describe().round(2))


taux_adoption_pct               float64
revenu_mensuel_recurrent_eur    float64
heures_usage_30j                float64
delai_reponse_support_h         float64
churn                             int64
dtype: object

       taux_adoption_pct  revenu_mensuel_recurrent_eur  heures_usage_30j  \
count            4750.00                       4850.00           4700.00   
mean               49.18                       3543.20             12.67   
std                25.68                       8795.24             17.48   
min                 0.00                          9.11              0.00   
25%                30.22                        141.72              0.50   
50%                50.00                        682.03              5.65   
75%                67.97                       2349.49             17.60   
max               100.00                      89402.85            170.20   

       delai_reponse_support_h  
count                  4500.00  
mean                     12.74  
s

## §4 — Catégorielles : casse, espaces et encodage [C3]

`plan` et `taille_entreprise` ne souffrent que d'espaces et de casse hétérogènes.
`secteur` cumule ce problème **et** une corruption d'encodage sur les valeurs
accentuées (ex. `Santé`, `Éducation` apparaissent tronquées ou avec des caractères de
remplacement `�`). Plutôt que de tenter de réparer l'encodage lui-même (source perdue,
non fiable), on normalise par correspondance de **sous-chaîne stable** — un motif ASCII
qui survit à la corruption, quelle que soit la casse.

In [5]:
df["plan"] = df["plan"].str.strip().str.title()
df["taille_entreprise"] = df["taille_entreprise"].str.strip().str.upper()

print("plan :", sorted(df["plan"].unique()))
print("taille_entreprise :", sorted(df["taille_entreprise"].unique()))


plan : ['Business', 'Enterprise', 'Pro', 'Starter']
taille_entreprise : ['ETI', 'GE', 'PME', 'TPE']


In [6]:
SECTOR_KEYWORDS = [
    ("TECH", "Tech"), ("FINANC", "Finance"), ("COMMERC", "Commerce"),
    ("SANT", "Santé"), ("INDUSTR", "Industrie"), ("PUBLIC", "Public"),
    ("DUC", "Éducation"),
]

def normalize_secteur(value):
    if pd.isna(value):
        return np.nan
    upper = value.strip().upper()
    for keyword, canonical in SECTOR_KEYWORDS:
        if keyword in upper:
            return canonical
    return "Inconnu"  # motif non reconnu — traçable plutôt que silencieusement perdu

df["secteur"] = df["secteur"].apply(normalize_secteur)
print(df["secteur"].value_counts(dropna=False))


secteur
Commerce     905
Tech         861
Finance      769
Industrie    666
Éducation    583
Santé        568
Public       398
NaN          250
Name: count, dtype: int64


Aucune valeur retombée dans `"Inconnu"` par défaut de reconnaissance (à vérifier dans
la sortie ci-dessus) — la correspondance par mot-clé couvre tous les cas rencontrés.
`pays` ne présente ni casse ni espaces incohérents (vérifié en TP1) — seules ses valeurs
manquantes restent à traiter en §6.

## §5 — Jointure avec le catalogue des plans [C3]

In [7]:
df_catalogue["plan"] = df_catalogue["plan"].str.strip().str.title()

before_cols = df.shape[1]
df = df.merge(df_catalogue, on="plan", how="left", validate="many_to_one")

n_non_joints = df["prix_mensuel_par_siege_eur"].isna().sum()
print(f"Colonnes ajoutées par la jointure : {df.shape[1] - before_cols}")
print(f"Lignes sans correspondance dans le catalogue : {n_non_joints}")


Colonnes ajoutées par la jointure : 5
Lignes sans correspondance dans le catalogue : 0


## §6 — Valeurs manquantes : taux et stratégie [C3]

In [8]:
na_rates = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print(na_rates[na_rates > 0])


commentaire_csm                 55.5
delai_reponse_support_h         10.0
csat                             8.0
heures_usage_30j                 6.0
retards_paiement_12m             5.0
secteur                          5.0
taux_adoption_pct                5.0
pays                             4.0
nb_integrations                  4.0
revenu_mensuel_recurrent_eur     3.0
dtype: float64


Stratégie retenue, colonne par colonne — pas d'imputation uniforme :

| Colonne | Taux NaN | Stratégie | Justification |
|---|---|---|---|
| `commentaire_csm` | 55.4% | Laissée telle quelle (NaN) | Texte libre, non utilisé comme feature numérique/catégorielle directe |
| `revenu_mensuel_recurrent_eur` | 3.0% | **Recalcul** via `sieges_souscrits × prix_mensuel_par_siege_eur` | Plus fiable qu'une médiane globale — le catalogue donne le vrai tarif par plan |
| `secteur`, `pays` | 5.0% / 4.0% | `"Inconnu"` explicite | Catégorielles : une valeur manquante n'est pas une moyenne, elle doit rester traçable |
| `taux_adoption_pct`, `heures_usage_30j`, `delai_reponse_support_h`, `csat`, `retards_paiement_12m`, `nb_integrations` | 4-10% | Médiane (documentée, calculée après nettoyage) | Numériques continues/ordinales, médiane robuste aux valeurs extrêmes déjà observées (ex. délai support jusqu'à 56h) |

In [9]:
# Recalcul prioritaire du MRR manquant à partir du catalogue (plus fiable qu'une médiane)
mrr_estime = df["sieges_souscrits"] * df["prix_mensuel_par_siege_eur"]
n_recalcules = df["revenu_mensuel_recurrent_eur"].isna().sum()
df["revenu_mensuel_recurrent_eur"] = df["revenu_mensuel_recurrent_eur"].fillna(mrr_estime)
print(f"revenu_mensuel_recurrent_eur recalculé pour {n_recalcules} lignes via sieges_souscrits x prix catalogue")

# Catégorielles : NaN explicite plutôt qu'imputation
for col in ["secteur", "pays"]:
    df[col] = df[col].fillna("Inconnu")

# Médiane documentée pour les numériques restants
MEDIAN_IMPUTE_COLS = [
    "taux_adoption_pct", "heures_usage_30j", "delai_reponse_support_h",
    "csat", "retards_paiement_12m", "nb_integrations",
]
medianes = {}
for col in MEDIAN_IMPUTE_COLS:
    med = df[col].median()
    medianes[col] = med
    df[col] = df[col].fillna(med)

print("\nMédianes utilisées :")
for col, val in medianes.items():
    print(f"  {col:28s} -> {val:.2f}")

print(f"\nNaN restants après traitement : {df.isna().sum().sum()} (hors commentaire_csm)")
print(df.drop(columns=["commentaire_csm"]).isna().sum().sum())


revenu_mensuel_recurrent_eur recalculé pour 150 lignes via sieges_souscrits x prix catalogue

Médianes utilisées :
  taux_adoption_pct            -> 50.00
  heures_usage_30j             -> 5.65
  delai_reponse_support_h      -> 10.50
  csat                         -> 3.00
  retards_paiement_12m         -> 0.00
  nb_integrations              -> 2.00

NaN restants après traitement : 2773 (hors commentaire_csm)
0


## §7 — Chargement dans PostgreSQL (nouvelle base `churn_saas_db`) [C3]

Décision actée en amont de ce TP : PostgreSQL est introduit dès maintenant, dans une
base **dédiée et indépendante** de `indusense_db` (projet `ML/`) — les deux cas d'usage
n'ont aucune raison de partager un schéma.

In [10]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

PG_HOST, PG_PORT = "localhost", 5432
PG_USER, PG_PASSWORD = "indusense_user", "ThEP@ssW0rd"
DB_NAME = "churn_saas_db"
TABLE_NAME = "clients_churn"

admin_url = URL.create("postgresql+psycopg2", username=PG_USER, password=PG_PASSWORD,
                        host=PG_HOST, port=PG_PORT, database="postgres")
admin_engine = create_engine(admin_url, isolation_level="AUTOCOMMIT")

with admin_engine.connect() as conn:
    exists = conn.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :name"), {"name": DB_NAME}
    ).fetchone()
    if not exists:
        conn.execute(text(f'CREATE DATABASE "{DB_NAME}"'))
        print(f"Base '{DB_NAME}' créée.")
    else:
        print(f"Base '{DB_NAME}' déjà existante — réutilisée.")


Base 'churn_saas_db' déjà existante — réutilisée.


In [11]:
db_url = URL.create("postgresql+psycopg2", username=PG_USER, password=PG_PASSWORD,
                    host=PG_HOST, port=PG_PORT, database=DB_NAME)
engine = create_engine(db_url)

df_to_load = df.copy()
df_to_load["date_souscription"] = df_to_load["date_souscription"].dt.date

df_to_load.to_sql(TABLE_NAME, engine, if_exists="replace", index=False, chunksize=1000)
# Note : la valeur de retour de to_sql() (nb de lignes du dernier chunk envoyé au
# driver) n'est pas fiable comme décompte total avec psycopg2 — la vérification
# réelle se fait par un COUNT(*) explicite en §8, pas ici.
print(f"Chargement envoyé pour {len(df_to_load):,} lignes -> {DB_NAME}.{TABLE_NAME} (vérification en §8)")


Chargement envoyé pour 5,000 lignes -> churn_saas_db.clients_churn (vérification en §8)


## §8 — Vérification finale

In [12]:
with engine.connect() as conn:
    n_rows = conn.execute(text(f"SELECT COUNT(*) FROM {TABLE_NAME}")).scalar()
    n_cols = conn.execute(text(
        "SELECT COUNT(*) FROM information_schema.columns WHERE table_name = :t"
    ), {"t": TABLE_NAME}).scalar()
    sample = pd.read_sql(f"SELECT * FROM {TABLE_NAME} LIMIT 3", engine)

print(f"Table {TABLE_NAME} : {n_rows:,} lignes x {n_cols} colonnes")
assert n_rows == len(df_to_load), "Écart entre le DataFrame nettoyé et la table chargée"
print("Cohérence DataFrame <-> table PostgreSQL : OK")
sample


Table clients_churn : 5,000 lignes x 34 colonnes
Cohérence DataFrame <-> table PostgreSQL : OK


,client_id,date_souscription,jour_souscription,secteur,pays,taille_entreprise,plan,anciennete_mois,sieges_souscrits,utilisateurs_actifs,taux_adoption_pct,connexions_30j,heures_usage_30j,fonctionnalites_total,fonctionnalites_utilisees,nb_integrations,derniere_connexion_jours,tickets_support_90j,delai_reponse_support_h,csat,retards_paiement_12m,revenu_mensuel_recurrent_eur,couleur_theme_interface,code_datacenter,groupe_experimentation,commentaire_csm,sante_compte_fin_periode,valeur_vie_client_eur,churn,prix_mensuel_par_siege_eur,fonctionnalites_incluses,sla_reponse_h,quota_stockage_go,support_dedie
0,CLI-002447,2024-01-31,mercredi,Finance,France,TPE,Starter,12,3,0,0.0,0,0.0,8,0,0.0,34,0,14.2,4.0,1.0,40.00,sombre,us-e1,control,None,38,689,0,12,8,24,10,Non
1,CLI-004125,2024-02-06,mardi,Santé,Belgique,TPE,Starter,12,3,1,33.3,2,0.4,8,2,1.0,0,5,27.9,3.0,0.0,31.14,violet,us-e1,control,None,45,485,0,12,8,24,10,Non
2,CLI-000087,2024-12-23,lundi,Éducation,France,TPE,Pro,1,10,2,20.0,1,0.1,16,0,0.0,25,2,17.7,1.0,2.0,171.02,clair,ap-s1,B,Mécontentement exprimé au support.,1,701,1,25,16,12,100,Non


## Journal de bord — Synthèse TP2

**Fait** :
- 35 doublons stricts supprimés (5035 → 5000 lignes).
- Dates unifiées sur 3 formats détectés (ISO, JJ/MM/AAAA, JJ Mon AAAA).
- 4 colonnes numériques stockées en texte corrigées (virgule décimale, suffixe `%`).
- Catégorielles normalisées — `secteur` par correspondance de mot-clé robuste à la
  corruption d'encodage observée, sans tentative de réparation de l'encodage lui-même.
- Jointure avec `catalogue_plans.csv` réussie sans perte de ligne.
- Valeurs manquantes traitées **par colonne**, pas par une règle uniforme — notamment
  le MRR manquant recalculé depuis le catalogue plutôt qu'imputé par une médiane
  aveugle.
- Données chargées dans `churn_saas_db.clients_churn` (PostgreSQL), vérifiées
  cohérentes avec le DataFrame en mémoire.

**Reste à faire** :
- TP3 — confirmer statistiquement le piège de fuite (`sante_compte_fin_periode`) et les
  leurres (`couleur_theme_interface`, `code_datacenter`, `groupe_experimentation`,
  `jour_souscription`), désormais sur des données propres et interrogeables en SQL.